# Time Series Pre-model Diagnostics

Module: Quantitative Methods and Financial Time Series

## Lesson summary

This lesson is a pre-model diagnostic workflow for one published Banxico FIX
series quoted as MXN per USD. It asks whether the price level and return transformation are
plausible modeling targets, whether raw returns retain linear dependence,
whether Gaussian shape is credible, and whether squared demeaned returns show
conditional heteroskedasticity. Model estimation belongs to later lessons
{cite}`box2015time,hamilton1994time,tsay2010analysis`.

## Learning objectives

By the end of this lesson, readers should be able to:

- distinguish a price-level unit-root screen from a return-level screen;
- read ACF and PACF plots with 95% confidence bands;
- label Ljung-Box on raw returns as a pre-model test;
- interpret Jarque-Bera and ARCH-LM without confusing their null hypotheses;
- distinguish expanding and rolling sample estimators from an EWMA filter;
- route ARIMA, GARCH, asymmetric-volatility, and VaR work to the lesson that
  owns each fitted-model decision.

## Prerequisites

[Lesson 2.1](2.1.time_series_1.ipynb) introduces price-to-return
transformations. This page uses no
fitted ARIMA or GARCH residuals, so every diagnostic below must remain
explicitly labeled as pre-model evidence.

## Stationarity and the modeling target

A process $\{y_t\}$ is weakly stationary when:

- $\mathbb E[y_t]=\mu$ is constant;
- $\operatorname{Var}(y_t)=\sigma^2<\infty$ is constant;
- $\operatorname{Cov}(y_t,y_{t-h})=\gamma_h$ depends only on lag $h$.

A random walk,

$$
y_t=y_{t-1}+\varepsilon_t,
$$

where $\{\varepsilon_t\}$ is a finite-variance white-noise innovation, has a
variance that grows with time. For a strictly positive level $P_t$, the
observed-to-observed log return is

$$
g_t=\log(P_t)-\log(P_{t-1}).
$$

A transformation can make stationarity more plausible, but no single test
proves that the transformed process is stationary in every future regime.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from statsmodels.tsa.stattools import acf as sample_acf

from src.market_data import banxico_daily_panel
from src.market_data_quality import log_returns
from src.market_risk import ewma_volatility
from src.module2_visuals import build_acf_pacf_figure, build_volatility_comparison
from src.time_series_diagnostics import (
    adf_report,
    arch_lm_report,
    jarque_bera_report,
    ljung_box_report,
)

## Reproducible source inventory

The dataset is the committed Banxico SIE snapshot, restricted to Banxico FIX
series SF43718. Selecting the series and dropping its missing rows preserves
only provider-published observations. There is no calendar reindexing and no
forward fill {cite}`banxicoSIE2025`.

In [ ]:
ANALYSIS_START = "2021-01-01"
ANALYSIS_END = "2026-06-05"
DIAGNOSTIC_LEVEL = 0.05
VOLATILITY_WINDOW = 63
EWMA_LAMBDA = 0.94

banxico_panel = banxico_daily_panel(start=ANALYSIS_START, end=ANALYSIS_END)
series_ids = banxico_panel.attrs.get("series_ids", {})
if series_ids.get("usd_mxn") != "SF43718":
    raise ValueError("The publication snapshot must map usd_mxn to Banxico SF43718.")

usd_mxn_fix = (
    pd.to_numeric(banxico_panel["usd_mxn"], errors="coerce")
    .dropna()
    .rename("usd_mxn_fix_mxn_per_usd")
)
if usd_mxn_fix.empty or not usd_mxn_fix.index.is_monotonic_increasing:
    raise ValueError("SF43718 must contain ordered published observations.")
if usd_mxn_fix.index.has_duplicates:
    raise ValueError("SF43718 must contain at most one published value per date.")
if not np.isfinite(usd_mxn_fix.to_numpy()).all() or (usd_mxn_fix <= 0).any():
    raise ValueError("SF43718 levels must be finite and strictly positive.")

interval_returns_pct = log_returns(usd_mxn_fix).mul(100).rename(
    "usd_mxn_fix_interval_log_return_pct"
)
demeaned_interval_returns_pct = (
    interval_returns_pct - interval_returns_pct.mean()
).rename(
    "demeaned_usd_mxn_fix_interval_log_return_pct"
)
observed_gap_days = usd_mxn_fix.index.to_series().diff().dt.days.dropna()
snapshot_vintage = banxico_panel.attrs.get("snapshot_generated_at", "not recorded")
source_note = (
    "Banxico SIE SF43718; observed "
    f"{usd_mxn_fix.index.min():%Y-%m-%d} to {usd_mxn_fix.index.max():%Y-%m-%d}; "
    f"snapshot {snapshot_vintage}; provider-dated observations, no forward fill"
)

source_inventory = pd.DataFrame(
    [
        ("provider", "Banco de México, SIE"),
        ("series", "SF43718 — Banxico FIX, MXN per USD"),
        ("snapshot vintage", snapshot_vintage),
        ("observed start", usd_mxn_fix.index.min().strftime("%Y-%m-%d")),
        ("observed end", usd_mxn_fix.index.max().strftime("%Y-%m-%d")),
        ("published level observations", f"{len(usd_mxn_fix):,}"),
        ("computed return observations", f"{len(interval_returns_pct):,}"),
        ("level unit", "MXN per USD"),
        ("return unit", "log-return percentage points per FIX publication interval"),
        (
            "observed calendar gaps",
            (
                f"min {int(observed_gap_days.min())}, "
                f"median {observed_gap_days.median():.0f}, "
                f"max {int(observed_gap_days.max())} days"
            ),
        ),
        ("diagnostic interval", "95% confidence level; 5% decision threshold"),
        ("observation policy", banxico_panel.attrs.get("observation_policy", "not recorded")),
        (
            "rights note",
            "provenance recorded; redistribution terms require external review",
        ),
    ],
    columns=["inventory item", "value"],
)
source_inventory

**How to read the inventory.**

The first published level has no return, so the return sample has one fewer
observation. Multi-day gaps are genuine gaps between published FIX values;
each computed return spans its two adjacent observed dates.

## ADF unit-root screen

The Augmented Dickey-Fuller test uses the null hypothesis that a unit root is
present. Rejecting that null at 5% is evidence against a unit root under this
specification and sample; failing to reject is not proof that the series is
a random walk {cite}`dickey1979distribution`.
With a constant deterministic term, the fitted auxiliary regression is

$$
\Delta y_t=a+\gamma y_{t-1}
+\sum_{i=1}^{k}\delta_i\Delta y_{t-i}+u_t,
\qquad H_0:\gamma=0.
$$

The helper selects $k$ by AIC and reports the resulting lag count. The level
and return screens use the same deterministic-term convention, but their
samples differ by the initial return lost to differencing.

In [ ]:
adf_results = pd.DataFrame(
    {
        "Banxico FIX level (MXN per USD)": adf_report(usd_mxn_fix, regression="c"),
        "Banxico FIX log return": adf_report(
            interval_returns_pct,
            regression="c",
        ),
    }
).T
adf_results["decision at 5%"] = np.where(
    adf_results["p_value"] < DIAGNOSTIC_LEVEL,
    "reject unit-root null",
    "do not reject unit-root null",
)
adf_table = adf_results[["statistic", "p_value", "used_lag", "nobs", "decision at 5%"]].copy()
adf_table[["used_lag", "nobs"]] = adf_table[["used_lag", "nobs"]].astype(int)
adf_table

The decision column is generated from the current snapshot and p-values. The
prose therefore does not embed a result that can become stale when the sample
changes.

## ACF and PACF with 95% confidence bands

The ACF at lag $h$ is

$$
\rho_h=\frac{\gamma_h}{\gamma_0}.
$$

The PACF at lag $h$ is the final coefficient $\phi_{hh}$ in the linear
projection of $g_t^{(\%)}$ on
$g_{t-1}^{(\%)},\ldots,g_{t-h}^{(\%)}$. It isolates the incremental linear
correlation at that lag after accounting for intermediate lags. The plotted
bands are approximate 95% confidence bands under a white-noise reference;
crossing a band is a screening signal, not automatic evidence for one ARIMA
order.

In [ ]:
acf_pacf_figure = build_acf_pacf_figure(
    interval_returns_pct,
    title="Banxico FIX (MXN per USD): pre-model ACF and PACF",
    lags=30,
    source=source_note,
    data_mode=banxico_panel.attrs.get("data_mode"),
)
display(acf_pacf_figure)
plt.close(acf_pacf_figure)

In [ ]:
acf_values = sample_acf(
    interval_returns_pct,
    nlags=30,
    fft=True,
    adjusted=False,
)
white_noise_acf_bound = 1.96 / np.sqrt(len(interval_returns_pct))
acf_screen = pd.DataFrame(
    {
        "lag": np.arange(1, len(acf_values)),
        "sample_acf": acf_values[1:],
        "approximate_95pct_bound": white_noise_acf_bound,
    }
).set_index("lag")
acf_screen["outside_approximate_band"] = (
    acf_screen["sample_acf"].abs() > white_noise_acf_bound
)
acf_screen.loc[acf_screen["outside_approximate_band"]]

**Output interpretation.**

The bounded table identifies any return-ACF lags outside the same approximate
white-noise reference used in the figure. Isolated crossings remain screening
evidence, not an ARIMA order rule; the joint Ljung-Box test below evaluates a
group of autocorrelations instead of selecting the most visually prominent
lag.

## Ljung-Box on raw returns

Ljung-Box tests whether a group of raw-return autocorrelations is jointly
zero {cite}`ljung1978measure`:

$$
Q=n(n+2)\sum_{j=1}^{k}\frac{\hat\rho_j^2}{n-j}.
$$

Because no mean model has been fitted here, `model_df=0`. Lesson 2.4 applies
the test to fitted ARIMA residuals with the appropriate parameter adjustment.

In [ ]:
ljung_box_raw = ljung_box_report(
    interval_returns_pct,
    lags=[5, 10, 20],
    model_df=0,
)
ljung_box_raw["decision at 5%"] = np.where(
    ljung_box_raw["lb_pvalue"] < DIAGNOSTIC_LEVEL,
    "reject joint zero-autocorrelation null",
    "do not reject joint zero-autocorrelation null",
)
ljung_box_raw

This is a linear-dependence screen on raw observations, not a residual
diagnostic and not a test of independence. Dependence may remain in squared
returns even when raw-return autocorrelation is weak.

## Jarque-Bera on raw returns

Jarque-Bera tests a Gaussian skewness-and-kurtosis restriction. Rejection
warns against unqualified Gaussian uncertainty statements; it does not by
itself select a mean or volatility model
{cite}`jarque1980efficient,tsay2010analysis`.
For sample skewness $S$, sample kurtosis $K$, and $n$ observations,

$$
JB=n\left(\frac{S^2}{6}+\frac{(K-3)^2}{24}\right),
\qquad H_0:S=0\ \text{and}\ K=3.
$$

In [ ]:
jarque_bera_raw = jarque_bera_report(interval_returns_pct)
jarque_bera_table = pd.DataFrame(
    [
        {
            "input": "raw Banxico FIX log returns",
            "statistic": jarque_bera_raw["statistic"],
            "p-value": jarque_bera_raw["p_value"],
            "decision at 5%": (
                "reject Gaussian shape null"
                if jarque_bera_raw["p_value"] < DIAGNOSTIC_LEVEL
                else "do not reject Gaussian shape null"
            ),
        }
    ]
)
jarque_bera_table

## ARCH-LM on demeaned returns

Engle's ARCH-LM test asks whether lagged squared innovations explain current
squared innovations. Before a fitted mean model exists, demeaned returns are a
transparent proxy and `model_df=0`. Lesson 2.2 subsequently estimates the
conditional variance {cite}`engle1982autoregressive`.
Its auxiliary regression and LM statistic are

$$
\widehat u_t^2=c+\sum_{i=1}^{k}\delta_i\widehat u_{t-i}^2+e_t,
\qquad LM=nR^2,
\qquad H_0:\delta_1=\cdots=\delta_k=0.
$$

In [ ]:
arch_lm_raw = arch_lm_report(
    demeaned_interval_returns_pct,
    lags=10,
    model_df=0,
)
arch_lm_table = pd.DataFrame(
    [
        {
            "input": "demeaned Banxico FIX log returns",
            "lags": int(arch_lm_raw["lags"]),
            "LM statistic": arch_lm_raw["lm_statistic"],
            "LM p-value": arch_lm_raw["lm_p_value"],
            "F statistic": arch_lm_raw["f_statistic"],
            "F p-value": arch_lm_raw["f_p_value"],
            "decision at 5%": (
                "reject no-ARCH null"
                if arch_lm_raw["lm_p_value"] < DIAGNOSTIC_LEVEL
                else "do not reject no-ARCH null"
            ),
        }
    ]
)
arch_lm_table

## Programmatic pre-model decision table

Each row below reads the statistic produced in the current run. The table
separates test nulls so that a small p-value is never translated into a
generic claim that “the series is significant.”

In [ ]:
diagnostic_decisions = pd.DataFrame(
    [
        {
            "diagnostic": "ADF — level",
            "input": "Banxico FIX, MXN per USD",
            "null hypothesis": "unit root",
            "p-value": adf_results.loc["Banxico FIX level (MXN per USD)", "p_value"],
            "decision at 5%": adf_results.loc[
                "Banxico FIX level (MXN per USD)", "decision at 5%"
            ],
            "next owner": "transformation choice on this page",
        },
        {
            "diagnostic": "ADF — return",
            "input": "log-return percentage points per FIX publication interval",
            "null hypothesis": "unit root",
            "p-value": adf_results.loc["Banxico FIX log return", "p_value"],
            "decision at 5%": adf_results.loc[
                "Banxico FIX log return", "decision at 5%"
            ],
            "next owner": "ARIMA candidates in Lesson 2.4",
        },
        {
            "diagnostic": "Ljung-Box — 10 lags",
            "input": "raw returns",
            "null hypothesis": "jointly zero autocorrelation through lag 10",
            "p-value": ljung_box_raw.loc[10, "lb_pvalue"],
            "decision at 5%": ljung_box_raw.loc[10, "decision at 5%"],
            "next owner": "fitted-residual check in Lesson 2.4",
        },
        {
            "diagnostic": "Jarque-Bera",
            "input": "raw returns",
            "null hypothesis": "Gaussian skewness and kurtosis",
            "p-value": jarque_bera_raw["p_value"],
            "decision at 5%": jarque_bera_table.loc[0, "decision at 5%"],
            "next owner": "innovation distribution in Lessons 2.2 and 2.5",
        },
        {
            "diagnostic": "ARCH-LM — 10 lags",
            "input": "demeaned returns",
            "null hypothesis": "no ARCH effects through lag 10",
            "p-value": arch_lm_raw["lm_p_value"],
            "decision at 5%": arch_lm_table.loc[0, "decision at 5%"],
            "next owner": "ARCH/GARCH comparison in Lesson 2.2",
        },
    ]
)
diagnostic_decisions

## Three distinct variance estimators and filters

The next-period quantities below use information available through $t-1$.
They summarize different information sets and are not interchangeable labels
for the same formula.

| Estimator | Variance formula | Information set and boundary |
| --- | --- | --- |
| Expanding sample variance | $\widehat\sigma_{t\mid t-1}^2=\frac{1}{t-2}\sum_{i=1}^{t-1}(g_i^{(\%)}-\bar g_{t-1}^{(\%)})^2$ | Uses every observed return through $t-1$; all available observations retain equal weight. |
| Rolling-window sample variance | $\widehat\sigma_{t\mid t-1,w}^2=\frac{1}{w-1}\sum_{i=t-w}^{t-1}(g_i^{(\%)}-\bar g_{t-1,w}^{(\%)})^2$ | Uses the latest $w$ returns through $t-1$; the oldest observation drops out when the window advances. |
| EWMA variance filter | $\sigma_{t\mid t-1}^2=(1-\lambda)(g_{t-1}^{(\%)})^2+\lambda\sigma_{t-1\mid t-2}^2$ | Uses a zero-mean return approximation; $0<\lambda<1$ controls decay and the initial variance is declared. |

Expanding and rolling estimators subtract their own prior-sample mean. The
EWMA path below uses a zero-mean return approximation and an initial variance
estimated from a separate 63-publication warm-up. It should not be described
as a rolling sample variance or as a calibrated GARCH forecast.

In [ ]:
lagged_returns_pct = interval_returns_pct.shift(1)
expanding_volatility_pct = lagged_returns_pct.expanding(
    min_periods=VOLATILITY_WINDOW
).std(ddof=1)
rolling_volatility_pct = lagged_returns_pct.rolling(
    VOLATILITY_WINDOW,
    min_periods=VOLATILITY_WINDOW,
).std(ddof=1)

ewma_warmup_returns = interval_returns_pct.iloc[:VOLATILITY_WINDOW]
ewma_filter_returns = interval_returns_pct.iloc[VOLATILITY_WINDOW:]
ewma_filtered_tail = ewma_volatility(
    ewma_filter_returns,
    lambda_=EWMA_LAMBDA,
    initial_variance=float(ewma_warmup_returns.var(ddof=1)),
)
ewma_volatility_pct = pd.Series(
    np.nan,
    index=interval_returns_pct.index,
    name="ewma_volatility_pct",
)
ewma_volatility_pct.loc[ewma_filtered_tail.index] = ewma_filtered_tail

volatility_estimator_paths = pd.DataFrame(
    {
        "Expanding through t-1": expanding_volatility_pct,
        f"Rolling {VOLATILITY_WINDOW} through t-1": rolling_volatility_pct,
        f"EWMA lambda={EWMA_LAMBDA:.2f}": ewma_volatility_pct,
    },
    index=interval_returns_pct.index,
)
assert volatility_estimator_paths.iloc[-1].notna().all()

In [ ]:
variance_estimator_figure = build_volatility_comparison(
    interval_returns_pct,
    volatility_estimator_paths,
    title="Banxico FIX (MXN per USD): prior-information volatility estimates",
    return_scale="percent",
    volatility_scale="percent",
    source=f"{source_note}; warm-up/window={VOLATILITY_WINDOW} publications",
    data_mode=banxico_panel.attrs.get("data_mode"),
)
display(variance_estimator_figure)
plt.close(variance_estimator_figure)

In [ ]:
latest_volatility_estimates = volatility_estimator_paths.iloc[-1].rename(
    "volatility_pct_per_fix_publication_interval"
).to_frame()
latest_volatility_estimates["target_FIX_publication"] = (
    interval_returns_pct.index[-1].strftime("%Y-%m-%d")
)
latest_volatility_estimates["information_through"] = (
    interval_returns_pct.index[-2].strftime("%Y-%m-%d")
)
latest_volatility_estimates

**Output interpretation.**

The paths share a return sample and per-publication-interval percentage scale,
but they react differently because their memory rules differ. The rolling
path drops one observation at each step, the expanding path changes gradually,
and EWMA assigns geometrically declining weights. None is annualized in this
figure, and visual similarity is not evidence that the estimators are
statistically equivalent.

## Bridges to fitted-model lessons

**ARIMA.** Lesson 2.4 owns candidate estimation, information-criterion
comparison, and residual Ljung-Box diagnostics. The raw-return tests here do
not select an ARIMA order.

**ARCH and GARCH.** Lesson 2.2 owns the controlled ARCH(3) versus GARCH(1,1)
fit, convergence guards, parameter tables, and annualized volatility paths.
ARCH-LM here only determines whether conditional-variance modeling is worth
investigating.

**GJR-GARCH and EGARCH.** Asymmetric terms require an explicit return sign.
Here $g_t=\Delta\log P_t$, where $P_t$ is MXN per USD: a positive return is
USD appreciation or peso depreciation, while a negative return is USD
depreciation or peso appreciation. A “negative-shock” indicator therefore
cannot be imported from the equity leverage story without translating that
FX sign convention.

**Dynamic VaR.** Lesson 2.5 owns heavy-tailed innovations, volatility
forecasting, and the positive-loss VaR sign contract. This page does not turn
a pre-model diagnostic into a risk forecast.

## Limitations

| Limitation | Why it matters |
| --- | --- |
| structural breaks | test and model parameters can change after crises or policy shifts |
| irregular publication gaps | an observed-to-observed return can span more than one calendar day |
| multiple lag screens | isolated band crossings or p-values need model-level confirmation |
| asymptotic tests | reported p-values can be sensitive to sample size and assumptions |
| outliers and heavy tails | extreme FX moves can dominate Gaussian diagnostics |
| pre-model scope | raw-return evidence is not evidence about fitted residual adequacy |

## Handoff

Carry the return transformation and pre-model decision table into
[Lesson 2.4](2.4.arima_diagnostic_workflow.ipynb) for conditional-mean
estimation. Carry the demeaned-return ARCH-LM evidence into
[Lesson 2.2](2.2.time_series_2.ipynb) for the controlled variance-model
comparison, then continue to
[Lesson 2.5](2.5.garch_volatility_risk_workflow.ipynb) for forecast and
positive-loss VaR decisions.